In [ ]:
import os
os.system('hostname')

In [ ]:
selected_model = 'name-of-best-bytestorm-model'
tracking_type = 'TT0.7-TB3-MT0.8-BB21'
our_work_name = 'ByteStorm'
setup = 'setup-name'

# rolling months
years = range(1980, 2020)
months = "rolling"

# lat and lon ranges of the domain
lon_range = [0, 360]
lat_range = [-70, 70]

# maximum distance to consider true the match
max_track_distance_matching = 300.0

# whether to store the figures or not
save_figures = True
compute_gif = False

# Get all the basins on which we compute metrics
basins = ['WNP', 'ENP', 'NI', 'SI', 'SP', 'NATL']
excluded_basins = ['SA', 'MED']

# output file extention
plot_kwargs = dict(dpi=100, bbox_inches='tight')
file_ext = 'pdf'

In [ ]:
from cartopy.mpl.ticker import (LongitudeFormatter, LatitudeFormatter)
import matplotlib.transforms as transforms
import matplotlib.ticker as mticker
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import cartopy.feature as cf
import cartopy.crs as ccrs

import pandas as pd
import xarray as xr
import numpy as np
import sys
import os

sys.path.append('../resources/library')
from tropical_cyclone.cyclone import (
    filter_basin, 
    filter_tracks, 
    compute_pod_and_far, 
    plot_storm_track_density, 
    compute_storm_transits, 
    signif, 
    remove_tcs_above_x_lat, 
    remove_tcs_originated_on_land, 
    plot_tracks, 
    remove_short_tracks, 
    convert_coords_to_format, 
)
from tropical_cyclone.utils import build_rolling_months
import dynamicopy

import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
# main dirs
data_src = 'PATH-TO/data'
dataset_src = os.path.join(data_src, 'dataset-folder')
bourdin_trackers_src = 'PATH-TO/zenodo_bourdin/trackers'
inference_src = os.path.join(data_src, 'inference-global')

# trackers source
ibtracs_src = os.path.join(data_src, 'ibtracs', 'filtered', 'global_ibtracs_main-tracks_6h_1980-2022_TS-NR-ET-MX-SS-DS.csv')
owz_src = os.path.join(bourdin_trackers_src, 'OWZ', 'tracks', 'ERA5_OWZ.csv')
track_src = os.path.join(bourdin_trackers_src, 'TRACK', 'ERA5_TRACK.csv')
uz_src = os.path.join(bourdin_trackers_src, 'UZ', 'tracks', 'ERA5_UZ.csv')
cnrm_src = os.path.join(bourdin_trackers_src, 'CNRM', 'tracks_vor15_res5_t1_pt-1_pw5_rel25.csv')

# our model
bytestorm_src = os.path.join(inference_src, f'{selected_model}/tracking_{tracking_type}-CLEAN.csv')

# directory to store images

out_dir = f'./notebooks/tracking/images/{setup}/{selected_model}'
if tracking_type is not None:
    out_dir += f'-{tracking_type}'
if months == "rolling":
    out_dir += f'-TEST-ROLLING-MONTHS'

In [ ]:
os.makedirs(out_dir, exist_ok=True)
out_dir

In [ ]:
INFO = {
    'ibtracs': {
        'color': 'tab:blue', 
        'label': 'IBTrACS', 
    }, 
    'owz': {
        'color': 'tab:pink', 
        'label': 'OWZ', 
    }, 
    'track': {
        'color': 'tab:green', 
        'label': 'TRACK', 
    }, 
    'uz': {
        'color': 'tab:orange', 
        'label': 'UZ', 
    }, 
    'cnrm': {
        'color': 'tab:purple', 
        'label': 'CNRM', 
    }, 
    'bytestorm': {
        'color': 'tab:red', 
        'label': f'{our_work_name}', 
    }, 
}

In [ ]:
ibtracs = pd.read_csv(ibtracs_src, index_col=0).rename(columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','SID':'track_id','BASIN':'basin'})[['time','lat','lon','track_id','basin','NATURE','WMO_WIND']]
owz = pd.read_csv(owz_src).rename(columns={' year':'year', ' month':'month', ' day':'day', ' hour':'hour',' lon':'lon',' lat':'lat'})
track = pd.read_csv(track_src, index_col=0)
uz = pd.read_csv(uz_src).rename(columns={' year':'year', ' month':'month', ' day':'day', ' hour':'hour',' lon':'lon',' lat':'lat'})
cnrm = pd.read_csv(cnrm_src).rename(columns={'Longitude':'lon', 'Latitude':'lat', 'Date':'time', 'ID':'track_id'})
bytestorm = pd.read_csv(bytestorm_src, index_col=0)
# Load Bourdin IBTrACS from dynamicopy
ibtracs_bo = dynamicopy.load_ibtracs()

In [ ]:
# convert iso_time to pandas datetime format
ibtracs['time'] = pd.to_datetime(ibtracs['time'])
owz['time'] = pd.to_datetime(owz[['year','month','day','hour']])
track['time'] = pd.to_datetime(track['time'])
uz['time'] = pd.to_datetime(uz[['year','month','day','hour']])
cnrm['time'] = pd.to_datetime(cnrm['time'])
bytestorm['time'] = pd.to_datetime(bytestorm['time'])
ibtracs_bo['time'] = pd.to_datetime(ibtracs_bo['time'])

In [ ]:
# Add basins information with dynamicopy
ibtracs['basin'] = dynamicopy.get_basin(ibtracs['lon'], ibtracs['lat'])
owz['basin'] = dynamicopy.get_basin(owz['lon'], owz['lat'])
track['basin'] = dynamicopy.get_basin(track['lon'], track['lat'])
uz['basin'] = dynamicopy.get_basin(uz['lon'], uz['lat'])
cnrm['basin'] = dynamicopy.get_basin(cnrm['lon'], cnrm['lat'])
bytestorm['basin'] = dynamicopy.get_basin(bytestorm['lon'], bytestorm['lat'])
ibtracs_bo['basin'] = dynamicopy.get_basin(ibtracs_bo['lon'], ibtracs_bo['lat'])

In [ ]:
if months == "rolling":
    year_and_month, years_dict = build_rolling_months(years[0], years[-1])
    test_years_dict = dict()
    for y,m in year_and_month:
        if y not in test_years_dict.keys():
            test_years_dict[y] = []
        test_years_dict[y] += [m]
    test_years_dict
    avail_dates = []
    dates = ibtracs['time'].unique()
    for date in dates:
        if date.year not in test_years_dict.keys(): continue
        if date.month in test_years_dict[date.year]:
            avail_dates.append(date)
    # for year in test_years_dict.keys():
    #     dates 
    #     dates = [date for date in dates if dates.year in test_years_dict.keys() and date.month in test_years_dict.keys()]
    dates = avail_dates
else:
    # Get the dates on which we perform evaluation
    dates = ibtracs['time'].unique()
    dates = [date for date in dates if date.year in years and date.month in months]
len(dates), len(ibtracs['time'].unique())

In [ ]:
# get intersection of all dates to be sure that each timesteps has been seen by EVERY tracker
# filter the tracks according to the procedure.
ibtracs = ibtracs[ibtracs['time'].isin(dates)]
owz = owz[owz['time'].isin(dates)]
track = track[track['time'].isin(dates)]
uz = uz[uz['time'].isin(dates)]
cnrm = cnrm[cnrm['time'].isin(dates)]
bytestorm = bytestorm[bytestorm['time'].isin(dates)]
ibtracs_bo = ibtracs_bo[ibtracs_bo['time'].isin(dates)]

In [ ]:
ibtracs = remove_short_tracks(ibtracs, min_track_count=12)
owz = remove_short_tracks(owz, min_track_count=12)
track = remove_short_tracks(track, min_track_count=12)
uz = remove_short_tracks(uz, min_track_count=12)
cnrm = remove_short_tracks(cnrm, min_track_count=12)
bytestorm = remove_short_tracks(bytestorm, min_track_count=12)
# ibtracs_bo = remove_short_tracks(ibtracs_bo, min_track_count=4)

In [ ]:
# Convert the coordinates to the [-180,180] format
coords_format = '[-180,180]'
ibtracs = convert_coords_to_format(ibtracs, format=coords_format)
ibtracs_bo = convert_coords_to_format(ibtracs_bo, format=coords_format)
owz = convert_coords_to_format(owz, format=coords_format)
track = convert_coords_to_format(track, format=coords_format)
uz = convert_coords_to_format(uz, format=coords_format)
cnrm = convert_coords_to_format(cnrm, format=coords_format)
bytestorm = convert_coords_to_format(bytestorm, format=coords_format)

# Top 5 Longest Tracks Comparison

In [ ]:
longest_ibtracs_track_ids = ibtracs.value_counts('track_id')[:5].reset_index()['track_id'].to_numpy()
longest_ibtracs_track_ids = [
    # '1990309N08167', # WNP
    # '2003286N09323', # NATL
    # '1999060S10089', # SI
    # '2008275N16245', # ENP
    # '2012346S14180', # SP
    # '1990309N08167', # 103 ['WNP']
    # '1997333N06194', # 89 ['ENP' 'WNP']
    # '2014066S09138', # 88 ['SP' 'SI']
    # '1994222N11267', # 87 ['ENP' 'WNP']
    # '1988285N09318', # 85 ['NATL' 'ENP']
    # '1999217N11265', # 85 ['ENP' 'WNP']
    # '2003286N09323', # 76 ['NATL']
    # '1999060S10089', # 74 ['SI']
    '1990309N08167', # 103 ['WNP']
    '1997333N06194', # 89 ['ENP' 'WNP']
    '2014066S09138', # 88 ['SP' 'SI']
    '1994222N11267', # 87 ['ENP' 'WNP']
    '1988285N09318', # 85 ['NATL' 'ENP']
    '1999217N11265', # 85 ['ENP' 'WNP']
    '2003286N09323', # 76 ['NATL']
    '1984214N12143', # 76 ['WNP']
    '2008275N16245', # 74 ['ENP']
    '1999060S10089', # 74 ['SI']
    '1998285N12149', # 74 ['WNP']
    '1991256N13171', # 72 ['WNP']
    '1981256N10150', # 72 ['WNP']
    '1996254N13138', # 72 ['WNP' 'ENP']
    '1989216N06159', # 71 ['WNP']
    '2004061S12072', # 69 ['SI']
    '1985162N05154', # 69 ['WNP']
    '1994212N11240', # 69 ['ENP' 'WNP']
    '1991257N16130', # 68 ['WNP']
    '1994215N30149', # 68 ['WNP']
]
ibtracs_longest_tracks = ibtracs[ibtracs['track_id'].isin(longest_ibtracs_track_ids)]
longest_ibtracs_track_ids

In [ ]:
ibtracs_longest_tracks.NATURE.unique(), ibtracs_longest_tracks.basin.unique()

In [ ]:
# get only TS storms
ibtracs_longest_tracks = ibtracs_longest_tracks[ibtracs_longest_tracks['NATURE'].isin(['TS', 'ET'])]

In [ ]:
owz_matches_lts, owz_res_lts = compute_pod_and_far(dynamicopy, owz, 'owz', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
track_matches_lts, track_res_lts = compute_pod_and_far(dynamicopy, track, 'track', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
uz_matches_lts, uz_res_lts = compute_pod_and_far(dynamicopy, uz, 'uz', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
cnrm_matches_lts, cnrm_res_lts = compute_pod_and_far(dynamicopy, cnrm, 'cnrm', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
bytestorm_matches_lts, bytestorm_res_lts = compute_pod_and_far(dynamicopy, bytestorm, our_work_name, ibtracs_longest_tracks, max_track_distance_matching, print_results=False)

In [ ]:
owz_matches_lts = pd.merge(owz_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
track_matches_lts = pd.merge(track_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
uz_matches_lts = pd.merge(uz_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
cnrm_matches_lts = pd.merge(cnrm_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
bytestorm_matches_lts = pd.merge(bytestorm_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')

In [ ]:
# load original ibtracs
ori_ibtracs = pd.read_csv(ibtracs_src, index_col=0).rename(columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','SID':'track_id'})[['time','lat','lon','track_id','NATURE','WMO_WIND','NAME']]

In [ ]:
# find the track ids and the corresponding name
track_id_and_name = pd.DataFrame({
    'track_id': longest_ibtracs_track_ids, 
    'name': [ori_ibtracs[ori_ibtracs['track_id']==tid].iloc[0]['NAME'] for tid in longest_ibtracs_track_ids] 
})
track_id_and_name

In [ ]:
owz_matches_lts = pd.merge(owz_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
track_matches_lts = pd.merge(track_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
uz_matches_lts = pd.merge(uz_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
cnrm_matches_lts = pd.merge(cnrm_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
bytestorm_matches_lts = pd.merge(bytestorm_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')

In [ ]:
def find_filename(date):
    return f'{date.year}_{date.month:02d}_{date.day:02d}_{date.hour:02d}.nc'

In [ ]:
def plot_tropical_cyclone_track_overlaid(
        ds, 
        owz, track, uz, cnrm, bytestorm, 
        owz_matches_lts, track_matches_lts, uz_matches_lts, cnrm_matches_lts, bytestorm_matches_lts, 
        selected_tc, selected_tc_track_id, selected_tc_ibtracs, driver, imtype='img', cmap='RdBu_r', outfile=None, 
):
    def get_tracks_by_ibtracs_track_id(tracks, matches, track_id, algo):
        track_match_id = matches[matches['id_ibtracs']==track_id][f'id_{algo}'].iloc[0]
        return tracks[tracks['track_id'] == track_match_id]

    def get_data(ds, variable, extent):
        lon_min, lon_max, lat_min, lat_max = extent
        if variable == 'msl':
            return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max)).min('time', skipna=True)[variable].load().data
        elif variable == 'vo_850':
            return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max)).max('time', skipna=True)[variable].load().data
        else:
            raise ValueError()

    if driver == 'vo_850':
        driver_title = 'Relative vorticity at 850mb maximum'
    elif driver == 'msl':
        driver_title = 'Mean Sea Level Pressure minimum'

    # colors = [
    #     'tab:blue', 
    #     'tab:orange', 
    #     'tab:green', 
    #     'tab:red', 
    #     'tab:purple', 
    #     'tab:cyan', 
    # ]
    # colors = plt.get_cmap('tab10').colors
    colors = plt.get_cmap('Set1').colors
    start_date, end_date = f'{selected_tc_ibtracs["time"].sort_values().iloc[0]}', f'{selected_tc_ibtracs["time"].sort_values().iloc[-1]}'

    # get the TC tracks for each tracker
    owz_tracks = get_tracks_by_ibtracs_track_id(owz, owz_matches_lts, selected_tc_track_id, 'owz')
    track_tracks = get_tracks_by_ibtracs_track_id(track, track_matches_lts, selected_tc_track_id, 'track')
    uz_tracks = get_tracks_by_ibtracs_track_id(uz, uz_matches_lts, selected_tc_track_id, 'uz')
    cnrm_tracks = get_tracks_by_ibtracs_track_id(cnrm, cnrm_matches_lts, selected_tc_track_id, 'cnrm')
    bytestorm_tracks = get_tracks_by_ibtracs_track_id(bytestorm, bytestorm_matches_lts, selected_tc_track_id, our_work_name)

    # get the extent of the data
    lat_min = np.round(np.min((selected_tc_ibtracs['lat'].min(), owz_tracks['lat'].min(), track_tracks['lat'].min(), uz_tracks['lat'].min(), cnrm_tracks['lat'].min(), bytestorm_tracks['lat'].min())) - 3)
    lat_max = np.round(np.max((selected_tc_ibtracs['lat'].max(), owz_tracks['lat'].max(), track_tracks['lat'].max(), uz_tracks['lat'].max(), cnrm_tracks['lat'].max(), bytestorm_tracks['lat'].max())) + 3)
    lon_min = np.round(np.min((selected_tc_ibtracs['lon'].min(), owz_tracks['lon'].min(), track_tracks['lon'].min(), uz_tracks['lon'].min(), cnrm_tracks['lon'].min(), bytestorm_tracks['lon'].min())) - 3)
    lon_max = np.round(np.max((selected_tc_ibtracs['lon'].max(), owz_tracks['lon'].max(), track_tracks['lon'].max(), uz_tracks['lon'].max(), cnrm_tracks['lon'].max(), bytestorm_tracks['lon'].max())) + 3)

    offset = -lon_min-96 if selected_tc == 'GUILLERMO' else 0
    image_extent = [lon_min, lon_max, lat_min, lat_max]

    width = (lon_max - lon_min) / lon_max
    height = (lat_max - lat_min) / lat_max
    ratio = width / height
    if ratio >= 0.4:
        figsize = (20, 6)
        cbar_kwargs = {'orientation': 'vertical', 'pad': 0.02}
    else:
        figsize = (10, 6)
        cbar_kwargs = {'orientation': 'vertical', 'pad': 0.02}

    # set map extent
    central_longitude = (lon_max - lon_min) / 2 if selected_tc not in ['DORA','JOHN','LI','VIOLET','NICHOLAS'] else 180
    proj_central_longitude = 0 if selected_tc in ['GUILLERMO'] else central_longitude

    fig = plt.figure(figsize=figsize)
    proj = ccrs.PlateCarree(central_longitude=proj_central_longitude)
    ax = plt.axes(projection=proj)

    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution='50m', lw=0.2)
    ax.add_feature(cf.LAND, facecolor='lightgrey', alpha=0.3)

    data = get_data(ds, driver, image_extent)

    fontdict = {'weight':'bold', 'size':14}
    # title_fontdict = {'size':18}
    ticksize = 12

    # plot tracks in each basin
    alpha = 0.5
    marker_size = 10.0
    transform = ccrs.PlateCarree()

    plt.title(f'{driver_title} of TC {selected_tc}\n({start_date} - {end_date})', fontsize = 18)

    # ibracs
    def plot_tracks(tracks, label, i):
        ax.plot(tracks['lon'], tracks['lat'], alpha=0.7, transform=transform, color=colors[i], zorder=90+i)
        ax.scatter(tracks['lon'], tracks['lat'], s=marker_size, marker='.', alpha=1.0, transform=transform, color=colors[i], label=label, zorder=90+i)

    plot_tracks(selected_tc_ibtracs, 'IBTrACS', 0)
    plot_tracks(owz_tracks, 'OWZ', 1)
    plot_tracks(track_tracks, 'TRACK', 2)
    plot_tracks(uz_tracks, 'UZ', 3)
    plot_tracks(cnrm_tracks, 'CNRM', 4)
    plot_tracks(bytestorm_tracks, our_work_name, 5)

    if imtype == 'img':
        im = ax.imshow(data, cmap=cmap, origin='upper', extent=image_extent, transform=ccrs.PlateCarree(), alpha=0.8)
    elif imtype == 'contourf':
        im = ax.contourf(data, cmap=cmap, origin='upper', extent=image_extent, transform=ccrs.PlateCarree(), alpha=0.8)
    
    fig.colorbar(im, ax=ax, **cbar_kwargs)

    if selected_tc in ['DORA','JOHN','LI','VIOLET','NICHOLAS']:
        # --- X-AXIS ---
        longitudes = np.arange(lon_min, lon_max+1, 10)
        lon_formatter = LongitudeFormatter(zero_direction_label=False)
        
        # 1. Imposta i tick numerici (proiettati)
        ax.set_xticks(longitudes - central_longitude)
        
        # 2. Genera i testi formattati (es. 170°W, 180, 170°E) partendo dai valori REALI
        labels_x = [lon_formatter(l) for l in longitudes]
        ax.set_xticklabels(labels_x, size=ticksize)
        ax.set_xlabel('Longitude [deg]', fontdict=fontdict)

        # --- Y-AXIS ---
        latitudes = np.arange(lat_min, lat_max+1, 10)
        lat_formatter = LatitudeFormatter()
        
        # 3. Stessa logica per la latitudine
        ax.set_yticks(latitudes)
        labels_y = [lat_formatter(l) for l in latitudes]
        ax.set_yticklabels(labels_y, size=ticksize)
        ax.set_ylabel('Latitude [deg]', fontdict=fontdict)

        # --- GRIDLINES ---
        # Importante: crs=proj significa che i locator devono essere già traslati (come stai facendo)
        gl = ax.gridlines(crs=proj, draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.xlocator = mticker.FixedLocator(longitudes - central_longitude)
        gl.ylocator = mticker.FixedLocator(latitudes)
    else:
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlocator = mticker.FixedLocator(np.arange(np.floor(lon_min), np.ceil(lon_max)+1, 10))
        gl.ylocator = mticker.FixedLocator(np.arange(np.floor(lat_min), np.ceil(lat_max)+1, 10))
        gl.xformatter = LongitudeFormatter()
        gl.yformatter = LatitudeFormatter()

    ax.set_xlabel('Longitude [deg]', fontdict=fontdict)
    ax.set_ylabel('Latitude [deg]', fontdict=fontdict)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', markerscale=5., edgecolor='black', framealpha=1, ncol=6, fontsize=14, bbox_to_anchor=(0.5,0.0))
    plt.tight_layout()
    plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
out_img_dir = os.path.join(out_dir, 'longest_tracks')
os.makedirs(out_img_dir, exist_ok=True)

In [ ]:
# ibtracs_longest_tracks['lon'] = ibtracs_longest_tracks['lon'] % 360
# owz['lon'] = owz['lon'] % 360
# track['lon'] = track['lon'] % 360
# uz['lon'] = uz['lon'] % 360
# cnrm['lon'] = cnrm['lon'] % 360
# bytestorm['lon'] = bytestorm['lon'] % 360

ibtracs_longest_tracks = convert_coords_to_format(ibtracs_longest_tracks, '[0,360]')
owz = convert_coords_to_format(owz, '[0,360]')
track = convert_coords_to_format(track, '[0,360]')
uz = convert_coords_to_format(uz, '[0,360]')
cnrm = convert_coords_to_format(cnrm, '[0,360]')
bytestorm = convert_coords_to_format(bytestorm, '[0,360]')

In [ ]:
ibtracs_longest_tracks # [ibtracs_longest_tracks['track_id'] == selected_tc_track_id]

In [ ]:
for selected_tc in track_id_and_name['name'].values: # ['ABBY', 'GUILLERMO', 'KEONI', 'JOHN', 'DORA']:
    try:
        selected_tc_track_id = track_id_and_name[track_id_and_name['name']==selected_tc]['track_id'].iloc[0]
        selected_tc_ibtracs = ibtracs_longest_tracks[ibtracs_longest_tracks['track_id'] == selected_tc_track_id]
        selected_tc_ibtracs['fname'] = selected_tc_ibtracs['time'].apply(find_filename)

        filenames = [os.path.join(dataset_src, fname) for fname in selected_tc_ibtracs['fname']]
        ds = xr.open_mfdataset(filenames)

        driver='vo_850'
        imtype='contourf'
        outfile = os.path.join(out_img_dir, f'{selected_tc}_{driver}_{imtype}.{file_ext}') if save_figures else None
        plot_tropical_cyclone_track_overlaid(
            ds, 
            owz, track, uz, cnrm, bytestorm, 
            owz_matches_lts, track_matches_lts, uz_matches_lts, cnrm_matches_lts, bytestorm_matches_lts, 
            selected_tc, selected_tc_track_id, selected_tc_ibtracs, driver=driver, imtype=imtype, cmap='Greys', outfile=outfile, 
        )
        driver='msl'
        imtype='contourf'
        outfile = os.path.join(out_img_dir, f'{selected_tc}_{driver}_{imtype}.{file_ext}') if save_figures else None
        plot_tropical_cyclone_track_overlaid(
            ds, 
            owz, track, uz, cnrm, bytestorm, 
            owz_matches_lts, track_matches_lts, uz_matches_lts, cnrm_matches_lts, bytestorm_matches_lts, 
            selected_tc, selected_tc_track_id, selected_tc_ibtracs, driver=driver, imtype=imtype, cmap='Greys_r', outfile=outfile, 
        )
    except:
        pass

In [ ]:
def plot_tropical_cyclone_track_overlaid(
        ds, bytestorm, bytestorm_matches_lts, 
        selected_tc, selected_tc_track_id, selected_tc_ibtracs, driver, imtype='img', cmap='RdBu_r', overlay=True, outfile=None, 
):
    def get_tracks_by_ibtracs_track_id(tracks, matches, track_id, algo):
        track_match_id = matches[matches['id_ibtracs']==track_id][f'id_{algo}'].iloc[0]
        return tracks[tracks['track_id'] == track_match_id]

    def get_data(ds, variable, extent):
        lon_min, lon_max, lat_min, lat_max = extent
        if variable == 'msl':
            return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max)).min('time', skipna=True)[variable].load().data
        elif variable == 'vo_850':
            return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max)).max('time', skipna=True)[variable].load().data
        else:
            raise ValueError()

    if driver == 'vo_850':
        driver_title = 'Relative vorticity at 850mb maximum'
    elif driver == 'msl':
        driver_title = 'Mean Sea Level Pressure minimum'

    # colors = [
    #     'tab:blue', 
    #     'tab:orange', 
    #     'tab:green', 
    #     'tab:red', 
    #     'tab:purple', 
    #     'tab:cyan', 
    # ]
    # colors = plt.get_cmap('tab10').colors
    colors = plt.get_cmap('Set1').colors
    start_date, end_date = f'{selected_tc_ibtracs["time"].sort_values().iloc[0]}', f'{selected_tc_ibtracs["time"].sort_values().iloc[-1]}'

    # get the TC tracks for each tracker
    bytestorm_tracks = get_tracks_by_ibtracs_track_id(bytestorm, bytestorm_matches_lts, selected_tc_track_id, our_work_name)

    # get the extent of the data
    lat_min = np.round(np.min((selected_tc_ibtracs['lat'].min(), bytestorm_tracks['lat'].min())) - 3)
    lat_max = np.round(np.max((selected_tc_ibtracs['lat'].max(), bytestorm_tracks['lat'].max())) + 3)
    lon_min = np.round(np.min((selected_tc_ibtracs['lon'].min(), bytestorm_tracks['lon'].min())) - 3)
    lon_max = np.round(np.max((selected_tc_ibtracs['lon'].max(), bytestorm_tracks['lon'].max())) + 3)
    
    offset = -lon_min-96 if selected_tc == 'GUILLERMO' else 0
    image_extent = [lon_min, lon_max, lat_min, lat_max]

    width = (lon_max - lon_min) / lon_max
    height = (lat_max - lat_min) / lat_max
    ratio = width / height
    if ratio >= 0.4:
        figsize = (20, 6)
        cbar_kwargs = {'orientation': 'vertical', 'pad': 0.02}
    else:
        figsize = (10, 6)
        cbar_kwargs = {'orientation': 'vertical', 'pad': 0.02}
    
    # set map extent
    central_longitude = (lon_max - lon_min) / 2 if selected_tc not in ['DORA','JOHN','LI','VIOLET','NICHOLAS'] else 180
    proj_central_longitude = 0 if selected_tc == 'GUILLERMO' else central_longitude

    fig = plt.figure(figsize=figsize)
    proj = ccrs.PlateCarree(central_longitude=proj_central_longitude)
    ax = plt.axes(projection=proj)

    ax.set_extent(image_extent, crs=ccrs.PlateCarree())
    ax.coastlines(resolution='50m', lw=0.2)
    ax.add_feature(cf.LAND, facecolor='lightgrey', alpha=0.3)

    data = get_data(ds, driver, image_extent)

    fontdict = {'weight':'bold', 'size':14}
    # title_fontdict = {'size':18}
    ticksize = 12

    # plot tracks in each basin
    alpha = 0.5
    marker_size = 10.0
    transform = ccrs.PlateCarree()

    plt.title(f'{driver_title} of TC {selected_tc}\n({start_date} - {end_date})', fontsize = 18)
    
    ax.plot(selected_tc_ibtracs['lon'], selected_tc_ibtracs['lat'], alpha=0.7, transform=transform, color='tab:blue', zorder=999)
    ax.scatter(selected_tc_ibtracs['lon'], selected_tc_ibtracs['lat'], s=marker_size, marker='.', alpha=1.0, transform=transform, color='tab:blue', label='IBTrACS', zorder=999)

    ax.plot(bytestorm_tracks['lon'], bytestorm_tracks['lat'], alpha=0.7, transform=transform, color='tab:red', zorder=999)
    ax.scatter(bytestorm_tracks['lon'], bytestorm_tracks['lat'], s=marker_size, marker='.', alpha=1.0, transform=transform, color='tab:red', label=our_work_name, zorder=999)

    if overlay:
        if imtype == 'img':
            im = ax.imshow(data, cmap=cmap, origin='upper', extent=image_extent, transform=ccrs.PlateCarree(), alpha=0.8)
        elif imtype == 'contourf':
            im = ax.contourf(data, cmap=cmap, origin='upper', extent=image_extent, transform=ccrs.PlateCarree(), alpha=0.8)
    
    fig.colorbar(im, ax=ax, **cbar_kwargs)

    if selected_tc in ['DORA','JOHN','LI','VIOLET','NICHOLAS']:
        # --- X-AXIS ---
        longitudes = np.arange(lon_min, lon_max+1, 10)
        lon_formatter = LongitudeFormatter(zero_direction_label=False)
        
        # 1. Imposta i tick numerici (proiettati)
        ax.set_xticks(longitudes - central_longitude)
        
        # 2. Genera i testi formattati (es. 170°W, 180, 170°E) partendo dai valori REALI
        labels_x = [lon_formatter(l) for l in longitudes]
        ax.set_xticklabels(labels_x, size=ticksize)
        ax.set_xlabel('Longitude [deg]', fontdict=fontdict)

        # --- Y-AXIS ---
        latitudes = np.arange(lat_min, lat_max+1, 10)
        lat_formatter = LatitudeFormatter()
        
        # 3. Stessa logica per la latitudine
        ax.set_yticks(latitudes)
        labels_y = [lat_formatter(l) for l in latitudes]
        ax.set_yticklabels(labels_y, size=ticksize)
        ax.set_ylabel('Latitude [deg]', fontdict=fontdict)

        # --- GRIDLINES ---
        # Importante: crs=proj significa che i locator devono essere già traslati (come stai facendo)
        gl = ax.gridlines(crs=proj, draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.xlocator = mticker.FixedLocator(longitudes - central_longitude)
        gl.ylocator = mticker.FixedLocator(latitudes)
    else:
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlocator = mticker.FixedLocator(np.arange(np.floor(lon_min), np.ceil(lon_max)+1, 10))
        gl.ylocator = mticker.FixedLocator(np.arange(np.floor(lat_min), np.ceil(lat_max)+1, 10))
        gl.xformatter = LongitudeFormatter()
        gl.yformatter = LatitudeFormatter()

    ax.set_xlabel('Longitude [deg]', fontdict=fontdict)
    ax.set_ylabel('Latitude [deg]', fontdict=fontdict)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', markerscale=5., edgecolor='black', framealpha=1, ncol=6, fontsize=14, bbox_to_anchor=(0.5,0.0))
    plt.tight_layout()
    if outfile is not None:
        plt.savefig(outfile, **plot_kwargs)
    else:
        plt.show()

In [ ]:
for selected_tc in track_id_and_name['name'].values:
    selected_tc_track_id = track_id_and_name[track_id_and_name['name']==selected_tc]['track_id'].iloc[0]
    selected_tc_ibtracs = ibtracs_longest_tracks[ibtracs_longest_tracks['track_id'] == selected_tc_track_id]
    selected_tc_ibtracs['fname'] = selected_tc_ibtracs['time'].apply(find_filename)
    
    filenames = [os.path.join(dataset_src, fname) for fname in selected_tc_ibtracs['fname']]
    ds = xr.open_mfdataset(filenames)
    
    driver='vo_850'
    imtype='contourf'
    outfile = os.path.join(out_img_dir, f'{selected_tc}_{driver}_{imtype}_bytestorm_only.{file_ext}') if save_figures else None
    plot_tropical_cyclone_track_overlaid(
        ds, bytestorm, bytestorm_matches_lts, 
        selected_tc, selected_tc_track_id, selected_tc_ibtracs, driver=driver, imtype=imtype, cmap='Greys', outfile=outfile, 
    )
    driver='msl'
    imtype='contourf'
    outfile = os.path.join(out_img_dir, f'{selected_tc}_{driver}_{imtype}_bytestorm_only.{file_ext}') if save_figures else None
    plot_tropical_cyclone_track_overlaid(
        ds, bytestorm, bytestorm_matches_lts, 
        selected_tc, selected_tc_track_id, selected_tc_ibtracs, driver=driver, imtype=imtype, cmap='Greys_r', outfile=outfile, 
    )